# Flash Floods KY Python Scripts

## Script #1: Moon & Sun Calculations

In [ ]:
# Import Libraries and Dependencies
from datetime import datetime
from pathlib import Path
import math
import ephem
import pandas as pd

In [ ]:
# Read dataset into dataframe, then inspect it
df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

df.head()

In [ ]:
# Create Astronomy Calculation Function
def calculate_local_astro_data(row: pd.Series) -> pd.Series:
    """Calculate sun and moon data for one row of the dataset."""
    
    try:
        # Get the event date
        date_obj = datetime.strptime(row["begin_date"], "%Y-%m-%d")
        
        # Create an observer using the event's coordinates
        observer = ephem.Observer()
        observer.lat = str(row["begin_lat"])
        observer.lon = str(row["begin_lon"])
        observer.date = date_obj
        
        # --------------------------------------------------
        # 1. SUN CALCULATIONS
        # --------------------------------------------------
        
        sun = ephem.Sun()
        sun.compute(observer)
        
        # Calculate sun altitude and azimuth
        sun_alt_deg = math.degrees(sun.alt)
        sun_az_deg = math.degrees(sun.az)
        
        # Reset observer date to the beginning of the day
        # to calculate sunrise and sunset
        observer.date = date_obj.date()
        
        try:
            sunrise = observer.next_rising(sun)
            sunrise_str = sunrise.datetime().strftime("%Y-%m-%d %H:%M:%S")
        except (ephem.CircumpolarError, ephem.AlwaysUpError):
            sunrise_str = "Sun always up"
        except ephem.NeverUpError:
            sunrise_str = "Sun never rises"
        
        try:
            sunset = observer.next_setting(sun)
            sunset_str = sunset.datetime().strftime("%Y-%m-%d %H:%M:%S")
        except (ephem.CircumpolarError, ephem.AlwaysUpError):
            sunset_str = "Sun always up"
        except ephem.NeverUpError:
            sunset_str = "Sun never sets"
        
        # --------------------------------------------------
        # 2. MOON CALCULATIONS
        # --------------------------------------------------
        
        # Reset observer to the event's exact date and time
        observer.date = date_obj
        
        moon = ephem.Moon()
        moon.compute(observer)
        
        # Calculate moon altitude and azimuth
        moon_alt_deg = math.degrees(moon.alt)
        moon_az_deg = math.degrees(moon.az)
        
        # Calculate moon illumination
        illumination = moon.phase / 100.0
        
        # Determine the moon's position within its lunar cycle
        prev_new = ephem.previous_new_moon(observer.date)
        next_new = ephem.next_new_moon(observer.date)
        
        lunation_age = (
            (observer.date - prev_new)
            * 29.53
            / (next_new - prev_new)
        )
        
        # Determine moon phase
        if illumination < 0.03:
            phase_name = "New Moon"
        
        elif illumination > 0.97:
            phase_name = "Full Moon"
        
        elif lunation_age < 14.77:
            if illumination < 0.45:
                phase_name = "Waxing Crescent"
            elif illumination < 0.55:
                phase_name = "First Quarter"
            else:
                phase_name = "Waxing Gibbous"
        
        else:
            if illumination > 0.55:
                phase_name = "Waning Gibbous"
            elif illumination > 0.45:
                phase_name = "Third Quarter"
            else:
                phase_name = "Waning Crescent"
        
        # Return the calculated astronomical data
        return pd.Series({
            "sun_altitude_deg": round(sun_alt_deg, 2),
            "sun_azimuth_deg": round(sun_az_deg, 2),
            "sunrise_utc": sunrise_str,
            "sunset_utc": sunset_str,
            "moon_altitude_deg": round(moon_alt_deg, 2),
            "moon_azimuth_deg": round(moon_az_deg, 2),
            "moon_phase_name": phase_name,
            "moon_illumination_pct": round(illumination * 100, 2)
        })
    
    except Exception as e:
        # Return error information if a calculation fails
        return pd.Series({
            "sun_altitude_deg": None,
            "sun_azimuth_deg": None,
            "sunrise_utc": f"Error: {str(e)}",
            "sunset_utc": f"Error: {str(e)}",
            "moon_altitude_deg": None,
            "moon_azimuth_deg": None,
            "moon_phase_name": f"Error: {str(e)}",
            "moon_illumination_pct": None
        })

In [ ]:
# Run astronomical calculations and display results
astro_data = df.apply(calculate_local_astro_data, axis=1)

astro_data.head()

In [ ]:
# Combine event_id with the calculated astronomy data
output_df = pd.concat(
    [df[["event_id"]], astro_data],
    axis=1
)

output_df.head()

In [ ]:
# Save dataframe as CSV
output_df.to_csv("../data/processed/flash_floods_ky_moon_sun_data.csv", index=False)

## Script #2: Chi-Square Goodness of Fit

In [ ]:
# Import Libraries and Dependencies
from pathlib import Path
import pandas as pd
from scipy.stats import chisquare

In [ ]:
# Read dataset into dataframe
df = pd.read_csv("../data/processed/flash_floods_ky_moon_sun_data.csv")

df.head()

In [ ]:
# Create bin boundaries 
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

In [ ]:
# Create bin labels
labels = [
    "0-10%", 
    "10-20%", 
    "20-30%", 
    "30-40%", 
    "40-50%", 
    "50-60%", 
    "60-70%", 
    "70-80%", 
    "80-90%", 
    "90-100%"
]

In [ ]:
# Assign each flash flooding event to a moon illumination bin
df["illumination_bin"] = pd.cut(
    df["moon_illumination_pct"], 
    bins=bins, 
    labels=labels, 
    include_lowest=True
)

df[["moon_illumination_pct", "illumination_bin"]].head(20)

In [ ]:
# Count observed flash flooding events
observed = df["illumination_bin"].value_counts().sort_index()

observed

In [ ]:
# Calculate expected flash flooding event counts
expected = [len(df) / len(observed)] * len(observed)

expected

In [ ]:
# Run Chi-Square Goodness of Fit test and display the results
chi2, p_value = chisquare(
    f_obs=observed, 
    f_exp=expected
)

print("Observed Counts:")
print(observed)

print("\nExpected Counts:")
print(expected)

print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6f}")

## Script #3: Oceanic Nino Index 

In [ ]:
# Import Libraries and Dependencies
import io
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

In [ ]:
# Read the ONI text file
with open("../data/raw/oni_backup.txt", "r", encoding="utf-8") as f: oni_text = f.read()

print(oni_text[:1000])

In [ ]:
# Parse ONI text into a dataframe, then check results
oni_df = pd.read_csv(
    io.StringIO(oni_text.strip()),
    sep=r"\s+",
    dtype={"YR": int, "ANOM": float}
)

oni_df.head()

In [ ]:
# Map ONI seasons to a central calendar month, then check results
season_to_month = {
    "DJF": 1,
    "JFM": 2,
    "FMA": 3,
    "MAM": 4,
    "AMJ": 5,
    "MJJ": 6,
    "JJA": 7,
    "JAS": 8,
    "ASO": 9,
    "SON": 10,
    "OND": 11,
    "NDJ": 12,
}

oni_df["month"] = oni_df["SEAS"].map(season_to_month)

oni_df.head()

In [ ]:
# Create a date column using the year and month, then check results
oni_df["date"] = pd.to_datetime(
    dict(
        year=oni_df["YR"],
        month=oni_df["month"],
        day=1
    )
)

oni_df.head()

In [ ]:
# Classify ENSO phase based on the ONI anomaly, then check results
oni_df["enso_phase"] = np.select(
    [
        oni_df["ANOM"] >= 0.5,
        oni_df["ANOM"] <= -0.5
    ],
    [
        "El Nino",
        "La Nina"
    ],
    default="Neutral"
)

oni_df.head()

In [ ]:
# Select the columns needed for the analysis and rename them for clarity, then check results
oni_df = oni_df[
    ["date", "SEAS", "ANOM", "enso_phase"]
].rename(
    columns={
        "SEAS": "oni_season",
        "ANOM": "oni_anomaly"
    }
)

oni_df.head()

In [ ]:
# Sort ONI data chronologically, then check results
oni_df = oni_df.sort_values("date").reset_index(drop=True)

oni_df.head()

In [ ]:
# Sort flash flood data chronologically, then check results
flood_df = flood_df.sort_values("begin_date").reset_index(drop=True)

flood_df[["event_id", "begin_date"]].head()

In [ ]:
# Check the data types of the columns before merging into one dataframe
print("flood_df begin_date:", flood_df["begin_date"].dtype)
print("oni_df date:", oni_df["date"].dtype)

In [ ]:
# Change flood_df begin_date from str datatype to datetime datatype
flood_df["begin_date"] = pd.to_datetime(
    flood_df["begin_date"]
)

print(flood_df["begin_date"].dtype)

In [ ]:
# Merge each flash flood event with the most recent ONI record occurring on or before the event date, then check results
enriched_df = pd.merge_asof(
    flood_df,
    oni_df,
    left_on="begin_date",
    right_on="date",
    direction="backward"
)

enriched_df.head()

In [ ]:
# Keep event_id and the ONI columns
enriched_df = enriched_df[
    [
        "event_id",
        "date",
        "oni_season",
        "oni_anomaly",
        "enso_phase"
    ]
]

enriched_df.head()

In [ ]:
# Save the merged DataFrame as a CSV file
enriched_df.to_csv("../data/processed/flash_floods_ky_oni_data.csv", index=False)

## Script #4: NLCD Landcover API

In [ ]:
# Import Libraries and Dependencies 
import os
from pathlib import Path
import pandas as pd
from arcgis.gis import GIS
from arcgis.raster import ImageryLayer
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

In [ ]:
# Load environment variables
load_dotenv()

In [ ]:
# Add configuration
API_KEY = os.environ.get("ARCGIS_API_KEY")
ARCGIS_URL = "https://arcgis.com"

In [ ]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

In [ ]:
# Define NLCD Layer IDs
LAYER_IDS = {
    "nlcd": "32e2ccc6416746a9a72b4d216813f84f",
    "elev": "58a541efc59545e6b7137f961d7de883",
    "imperv": "6df535f263dd44f489365eed49461a38",
}

In [ ]:
# Define NLCD classes
NLCD_CLASSES = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

In [ ]:
# Connect to ArcGIS
def initialize_layers(api_key: str) -> tuple:
    """Connects to the ArcGIS GIS API with strict SSL validation
    and returns the three imagery layers."""

    print("Connecting to ArcGIS Living Atlas layers...")

    gis = GIS(
        ARCGIS_URL,
        api_key=api_key,
        verify_cert=True
    )

    layers = {
        name: ImageryLayer(
            gis.content.get(item_id).url,
            gis=gis
        )
        for name, item_id in LAYER_IDS.items()
    }

    print("All layers connected successfully.")

    return (
        layers["nlcd"],
        layers["elev"],
        layers["imperv"]
    )

In [ ]:
# Query One Imagery Layer
def query_layer_value(layer: ImageryLayer, geom: dict):
    """Queries an imagery layer at a point geometry
    and returns its pixel value, or None on failure."""

    try:
        response = layer.identify(
            geometry=geom,
            return_pixel_values=True
        )

        return response.get("value", None)

    except Exception:
        return None

In [ ]:
# Process One Flash Flood Event
def process_single_row(idx, row, layers):
    """Worker function to process a single row's
    spatial variables concurrently."""

    nlcd_layer, elev_layer, imp_layer = layers

    mid_lat = (row["begin_lat"] + row["end_lat"]) / 2
    mid_lon = (row["begin_lon"] + row["end_lon"]) / 2

    if pd.isna(mid_lat) or pd.isna(mid_lon):
        return idx, (None, None, None, None)

    geom = {
        "x": mid_lon,
        "y": mid_lat,
        "spatialReference": {"wkid": 4326}
    }

    val_nlcd = query_layer_value(nlcd_layer, geom)
    val_elev = query_layer_value(elev_layer, geom)
    val_imp = query_layer_value(imp_layer, geom)

    c_code = int(val_nlcd) if val_nlcd is not None else None

    c_class = (
        NLCD_CLASSES.get(c_code, "Unknown")
        if c_code is not None
        else None
    )

    elevation = (
        float(val_elev)
        if val_elev is not None
        else None
    )

    impervious = (
        int(val_imp)
        if val_imp is not None
        else None
    )

    return idx, (
        c_code,
        c_class,
        elevation,
        impervious
    )

In [ ]:
# Process all flash flooding events within the dataframe
def fetch_geospatial_attributes(
    df: pd.DataFrame,
    layers: tuple,
    max_workers: int = 20,
) -> pd.DataFrame:

    """Fetches geospatial attributes for each row
    and joins them onto the dataframe."""

    print(
        f"Starting batch execution using "
        f"{max_workers} concurrent threads..."
    )

    cols = [
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]

    results = {}

    total_rows = len(df)

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {
            executor.submit(
                process_single_row,
                idx,
                row,
                layers
            ): idx

            for idx, row in df.iterrows()
        }

        for completed, future in enumerate(
            as_completed(futures), 1
        ):

            idx, values = future.result()

            results[idx] = values

            if completed % 100 == 0 or completed == total_rows:
                print(
                    f"Progress: {completed}/{total_rows} "
                    f"rows extracted "
                    f"({completed/total_rows*100:.1f}%)"
                )

    result_df = pd.DataFrame.from_dict(
        results,
        orient="index",
        columns=cols
    )

    return df.join(result_df)

In [ ]:
# Connect to ArcGIS
atlas_layers = initialize_layers(API_KEY)

In [ ]:
# Run geospatial extraction
processed_df = fetch_geospatial_attributes(
    flood_df,
    atlas_layers,
    max_workers=25
)

In [ ]:
# Inspect processed dataset
processed_df.head()

In [ ]:
# Keep only event_id and NLCD columns
processed_df = processed_df[
    [
        "event_id",
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]
]

processed_df.head()

In [ ]:
# Save processed dataset as CSV file
processed_df.to_csv("../data/processed/flash_floods_ky_nlcd_data.csv", index=False)

## Script #5: Weather API

In [ ]:
# Import Libraries and Dependencies
import os
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv

In [ ]:
# Load environment variables
load_dotenv()

In [ ]:
# Add configuration
WEATHER_API_URL = "https://api.weatherapi.com/v1/history.json"
WEATHER_API_KEY = os.environ["WEATHER_API_KEY"]

In [ ]:
# Read in dataset as dataframe
flash_flood_df = pd.read_csv("../data/processed/flash_floods_ky_2015_2025_cleaned.csv")

flash_flood_df.head()

In [ ]:
# Define weather fields
WEATHER_FIELDS = [
    "maxtemp_f",
    "mintemp_f",
    "avgtemp_f",
    "maxwind_mph",
    "totalprecip_in",
    "avgvis_miles",
    "avghumidity",
    "uv",
    "daily_will_it_rain",
    "daily_chance_of_rain",
    "condition_text",
    "condition_code"
]

In [ ]:
# Prepare dates to match WeatherAPI dates
flash_flood_df["begin_date_cleaned"] = (
    pd.to_datetime(flash_flood_df["begin_date"])
    .dt.strftime("%Y-%m-%d")
)

flash_flood_df[["begin_date", "begin_date_cleaned"]].head()

In [ ]:
# Define WeatherAPI function
def get_historical_weather_row(api_key, location, date_str):
    params = {
        "key": api_key,
        "q": f"{location}, Kentucky, USA",
        "dt": date_str
    }

    result = {field: None for field in WEATHER_FIELDS}

    try:
        response = requests.get(
            WEATHER_API_URL,
            params=params,
        )

        if response.status_code != 200:
            print(
                f"Skipping {location} on {date_str}: "
                f"API Status {response.status_code}"
            )
            print(response.text)
            return result

        forecast_day = (
            response.json()
            ["forecast"]["forecastday"][0]["day"]
        )

        for field in WEATHER_FIELDS:
            result[field] = forecast_day.get(field)

    except requests.exceptions.Timeout:
        print(f"Timeout for {location} on {date_str}")

    except requests.exceptions.RequestException as e:
        print(
            f"Request error for {location} on {date_str}: {e}"
        )

    except KeyError as e:
        print(
            f"Missing expected data for {location} on {date_str}: {e}"
        )

    except Exception as e:
        print(
            f"Error processing {location} on {date_str}: {e}"
        )

    return result

In [ ]:
# Call WeatherAPI function
collected_weather = []

for idx, row in flash_flood_df.iterrows():

    location = row["begin_location"]
    date_str = row["begin_date_cleaned"]

    print(
        f"Processing row {idx + 1}/{len(flash_flood_df)}: "
        f"{location} on {date_str}"
    )

    weather_data = get_historical_weather_row(
        WEATHER_API_KEY,
        location,
        date_str
    )

    collected_weather.append(weather_data)

    time.sleep(1)

In [ ]:
# Create weather dataframe
weather_df = pd.DataFrame(collected_weather)

weather_df.head()

In [ ]:
# Combine weather dataframe with orignal dataframe, then check results
flash_flood_df = pd.concat(
    [
        flash_flood_df.reset_index(drop=True),
        weather_df.reset_index(drop=True)
    ],
    axis=1
)

flash_flood_df.head()

In [ ]:
# Remove temporary date column, then check results
flash_flood_df = flash_flood_df.drop(
    columns=["begin_date_cleaned"]
)

flash_flood_df.head()

In [ ]:
# Save dataframe as CSV
flash_flood_df.to_csv("data/processed/flash_floods_ky_weather_conditions.csv",index=False)